# SupermarketDemand — Del dato bruto al dato utilizable

**Entrega 2 · Datos y preparación**

Este notebook recorre el flujo **Bronze → Silver → Gold** con datos reales de
[FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)
(licencia CC BY 4.0): ventas de productos frescos por tienda y día, con la venta y el
stock hora a hora.

```
Hugging Face ──▶ BRONZE ──▶ SILVER ──▶ GOLD
                 tal cual    limpio     features + objetivo
                 llega       y tipado   para el modelo
```

**Qué se quiere predecir (Gold):** la venta de **mañana** de un producto en una tienda,
para ayudar a decidir cuánto pedir y así reducir desperdicio y roturas de stock.

> Si usas Google Colab, ejecuta primero esta línea (quita el `#`):

In [1]:
%pip install -q datasets pyarrow pandas

In [2]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# --- Configuración -----------------------------------------------------------
DATASET_ID = "Dingdong-Inc/FreshRetailNet-50K"
SAMPLE_SERIES = 20      # series tienda-producto que se guardan como muestra para el repo
SEED = 42               # semilla fija: la muestra es siempre la misma
STORES = None           # None = todas las tiendas. Si falta memoria: por ejemplo [0, 1, 2, 3]

# Rutas (funciona ejecutando desde la raíz del repo o desde notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
SAMPLE = ROOT / "data" / "sample"
SILVER = ROOT / "data" / "silver"
GOLD = ROOT / "data" / "gold"
for folder in (RAW, SAMPLE, SILVER, GOLD):
    folder.mkdir(parents=True, exist_ok=True)

KEY = ["store_id", "product_id"]      # una serie temporal = una tienda + un producto
HOURS = 24

---
## 1. Bronze — el dato tal como llega

**Objetivo:** conseguir el dato de forma reproducible y guardarlo **sin tocarlo**.

- Se descarga de Hugging Face (`train` y `eval`) y se guarda en Parquet en `data/raw/`.
- Si el fichero ya existe, no se vuelve a descargar.
- Se guarda un `bronze_manifest.json` con la fuente, la licencia y la fecha de descarga.
- Como los datos completos no se suben al repositorio, se guarda una **muestra pequeña**
  (series completas elegidas al azar con semilla fija) en `data/sample/`.

In [3]:
train_path = RAW / "train.parquet"

if not train_path.exists():
    from datasets import load_dataset   # solo hace falta si hay que descargar

    dataset = load_dataset(DATASET_ID)
    manifest = {
        "source": DATASET_ID,
        "license": "CC BY 4.0",
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "splits": {},
    }
    for name, split in dataset.items():
        path = RAW / f"{name}.parquet"
        split.to_parquet(str(path))
        manifest["splits"][name] = {"rows": len(split), "file": path.name}
    (RAW / "bronze_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest, indent=2))
else:
    print("Bronze ya existe, no se descarga de nuevo:", train_path)

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/eval.parquet: reconstructing file:   0%|          |  0.00B / 8.44MB            

data/eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/24 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

{
  "source": "Dingdong-Inc/FreshRetailNet-50K",
  "license": "CC BY 4.0",
  "downloaded_at_utc": "2026-11-02T12:52:17+00:00",
  "splits": {
    "train": {
      "rows": 4500000,
      "file": "train.parquet"
    },
    "eval": {
      "rows": 350000,
      "file": "eval.parquet"
    }
  }
}


In [4]:
# Muestra reproducible: 20 series (tienda-producto) completas, elegidas al azar
sample_path = SAMPLE / "freshretailnet_sample.parquet"

if not sample_path.exists():
    pairs = pq.read_table(train_path, columns=KEY).to_pandas().drop_duplicates()
    chosen = pairs.sample(n=min(SAMPLE_SERIES, len(pairs)), random_state=SEED)
    raw_keys = pq.read_table(train_path, columns=KEY).to_pandas()
    keep = raw_keys.merge(chosen.assign(_keep=True), on=KEY, how="left")["_keep"].fillna(False).to_numpy()
    table = pq.read_table(train_path).filter(pa.array(keep))
    pq.write_table(table, sample_path)
    print(f"Muestra: {table.num_rows:,} filas ({len(chosen)} series) -> {sample_path.name}")
else:
    print("La muestra ya existe:", sample_path)

/tmp/ipykernel_2313/1674063192.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  keep = raw_keys.merge(chosen.assign(_keep=True), on=KEY, how="left")["_keep"].fillna(False).to_numpy()


Muestra: 1,800 filas (20 series) -> freshretailnet_sample.parquet


### Cómo es el dato bruto

Una fila = **una tienda, un producto y un día**. Fíjate en que `hours_sale` y
`hours_stock_status` son **listas de 24 valores** (uno por hora) y que `stock_hour6_22_cnt`
es un resumen que ya trae la fuente.

In [5]:
raw_peek = pq.read_table(train_path).slice(0, 3).to_pandas()
print(raw_peek.dtypes)
raw_peek.drop(columns=["hours_sale", "hours_stock_status"])

city_id                  int64
store_id                 int64
management_group_id      int64
first_category_id        int64
second_category_id       int64
third_category_id        int64
product_id               int64
dt                      object
sale_amount            float64
hours_sale              object
stock_hour6_22_cnt       int32
hours_stock_status      object
discount               float64
holiday_flag             int32
activity_flag            int32
precpt                 float64
avg_temperature        float64
avg_humidity           float64
avg_wind_level         float64
dtype: object


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,stock_hour6_22_cnt,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,0,5,6,65,38,2024-03-28,0.1,0,1.0,0,0,1.6999,15.48,73.54,1.97
1,0,0,0,5,6,65,38,2024-03-29,0.1,1,1.0,0,0,3.0190,15.08,76.56,1.71
2,0,0,0,5,6,65,38,2024-03-30,0.0,0,1.0,1,0,2.0942,15.91,76.47,1.73


In [6]:
# Una fila completa, con sus 24 horas de venta y de stock
row = raw_peek.iloc[0]
print("Venta del día:", row["sale_amount"], "| venta por hora:", list(row["hours_sale"]))
print("Stock por hora (1 = sin stock):", list(row["hours_stock_status"]))

Venta del día: 0.1 | venta por hora: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.1), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
Stock por hora (1 = sin stock): [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]


---
## 2. Silver — limpio, tipado y aplanado

**Objetivo:** que cada fila sea **una tienda-producto-día** fiable y con los tipos correctos.

| Paso | Qué hace | Por qué |
|---|---|---|
| Esquema | Comprueba que están las 19 columnas esperadas | Falla pronto y con un mensaje claro si la fuente cambia |
| Arrays horarios | Descarta filas con listas que no tengan 24 valores o con nulos | No se pueden agregar de forma fiable |
| Nulos | Descarta filas sin tienda, producto, fecha o venta | Sin clave u objetivo no sirven |
| Duplicados | Una fila por tienda-producto-día | Evita contar dos veces el mismo día |
| Ventas negativas | Se descartan | Una demanda observada no puede ser negativa |
| Tipos | Fechas, enteros, `float32` | Menos memoria y operaciones correctas |
| Aplanado horario | De las 24 horas se derivan: hora pico y horas sin stock | Resume lo horario en columnas fáciles de usar |

Todo lo que se descarta **se cuenta** en un informe, para poder justificarlo.

In [7]:
RAW_COLUMNS = [
    "city_id", "store_id", "management_group_id",
    "first_category_id", "second_category_id", "third_category_id",
    "product_id", "dt", "sale_amount", "hours_sale",
    "stock_hour6_22_cnt", "hours_stock_status",
    "discount", "holiday_flag", "activity_flag",
    "precpt", "avg_temperature", "avg_humidity", "avg_wind_level",
]
RENAME = {
    "dt": "date", "sale_amount": "sales",
    "first_category_id": "category_1", "second_category_id": "category_2",
    "third_category_id": "category_3",
    "stock_hour6_22_cnt": "stockout_hours_6_22_raw",
    "holiday_flag": "holiday", "activity_flag": "activity",
    "precpt": "precipitation", "avg_temperature": "temperature",
    "avg_humidity": "humidity", "avg_wind_level": "wind_level",
}

# Leer Bronze (opcionalmente solo algunas tiendas) y comprobar el esquema
filters = [("store_id", "in", STORES)] if STORES else None
table = pq.read_table(train_path, filters=filters)

missing = [c for c in RAW_COLUMNS if c not in table.column_names]
assert not missing, f"Faltan columnas en la fuente: {missing}"

report = {"rows_in": table.num_rows, "dropped": {}}
print(f"Filas de entrada: {report['rows_in']:,}")

Filas de entrada: 4,500,000


In [8]:
# Arrays horarios: solo filas con exactamente 24 valores en las dos listas
ok = pc.and_(
    pc.equal(pc.list_value_length(table["hours_sale"]), HOURS),
    pc.equal(pc.list_value_length(table["hours_stock_status"]), HOURS),
)
ok = pc.fill_null(ok, False)
report["dropped"]["hours_array_not_24"] = int(table.num_rows - pc.sum(ok).as_py())
table = table.filter(ok)

# Aplanar: lista de 24 valores por fila -> matriz (filas, 24)
def to_matrix(column):
    flat = column.combine_chunks().flatten()
    return flat.to_numpy(zero_copy_only=False).astype("float32").reshape(-1, HOURS)

hours_sale = to_matrix(table["hours_sale"])
hours_stock = to_matrix(table["hours_stock_status"])

df = table.select([c for c in RAW_COLUMNS if c not in ("hours_sale", "hours_stock_status")]).to_pandas()

# Agregados diarios a partir de lo horario (el detalle horario sigue en Bronze)
df["hours_sales_sum"] = hours_sale.sum(axis=1)
df["peak_hour"] = np.where(hours_sale.sum(axis=1) > 0, hours_sale.argmax(axis=1), -1).astype("int8")
df["stockout_hours_total"] = hours_stock.sum(axis=1).astype("int8")
df["stockout_hours_6_21"] = hours_stock[:, 6:22].sum(axis=1).astype("int8")   # horas 6..21
has_nan_hours = np.isnan(hours_sale).any(axis=1) | np.isnan(hours_stock).any(axis=1)

df = df.rename(columns=RENAME)
df.head()

,city_id,store_id,management_group_id,category_1,category_2,category_3,product_id,date,sales,stockout_hours_6_22_raw,...,holiday,activity,precipitation,temperature,humidity,wind_level,hours_sales_sum,peak_hour,stockout_hours_total,stockout_hours_6_21
0,0,0,0,5,6,65,38,2024-03-28,0.1,0,...,0,0,1.6999,15.48,73.54,1.97,0.1,7,0,0
1,0,0,0,5,6,65,38,2024-03-29,0.1,1,...,0,0,3.0190,15.08,76.56,1.71,0.1,6,3,1
2,0,0,0,5,6,65,38,2024-03-30,0.0,0,...,1,0,2.0942,15.91,76.47,1.73,0.0,-1,6,0
3,0,0,0,5,6,65,38,2024-03-31,0.1,11,...,1,0,1.5618,16.13,77.40,1.76,0.1,8,13,11
4,0,0,0,5,6,65,38,2024-04-01,0.2,8,...,0,0,3.5386,15.37,78.26,1.25,0.2,6,15,8


In [9]:
# Tipos
df["date"] = pd.to_datetime(df["date"], errors="coerce")
for col in ["sales", "discount", "precipitation", "temperature", "humidity", "wind_level"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

def drop_rows(mask, reason):
    # Elimina las filas marcadas y apunta cuántas fueron
    global df, has_nan_hours
    mask = np.asarray(mask, dtype=bool)
    report["dropped"][reason] = int(mask.sum())
    df, has_nan_hours = df[~mask], has_nan_hours[~mask]

drop_rows(df[["store_id", "product_id", "date", "sales"]].isna().any(axis=1), "null_in_key_or_sales")
drop_rows(has_nan_hours, "nan_inside_hours_arrays")
drop_rows(df.duplicated(subset=KEY + ["date"], keep="first"), "duplicate_store_product_day")
drop_rows(df["sales"] < 0, "negative_sales")

# Tipos finales
for col in ["city_id", "store_id", "management_group_id", "product_id",
            "category_1", "category_2", "category_3"]:
    df[col] = df[col].astype("int32")
for col in ["holiday", "activity"]:
    df[col] = df[col].astype("Int8")
for col in ["sales", "discount", "precipitation", "temperature", "humidity", "wind_level", "hours_sales_sum"]:
    df[col] = df[col].astype("float32")

report["dropped"]

{'hours_array_not_24': 0,
 'null_in_key_or_sales': 0,
 'nan_inside_hours_arrays': 0,
 'duplicate_store_product_day': 0,
 'negative_sales': 0}

### Controles de calidad

Estos controles **no filtran nada**: miden si los supuestos sobre la fuente se cumplen.
Si alguno sale muy lejos de 0 %, hay que revisarlo antes de modelar.

In [10]:
qa = {
    # ¿la venta diaria coincide con la suma de las ventas horarias?
    "venta_diaria_distinta_de_suma_horaria": float((df["sales"] - df["hours_sales_sum"]).abs().gt(0.1).mean()),
    # ¿el resumen de stock de la fuente coincide con nuestro recuento de las horas 6..21?
    "stock_6_22_fuente_distinto_de_recuento_propio": float(
        (df["stockout_hours_6_22_raw"].astype("float32") != df["stockout_hours_6_21"].astype("float32")).mean()),
    # ¿el descuento es un factor entre 0 y 1?
    "descuento_fuera_de_(0,1]": float((~df["discount"].between(0, 1, inclusive="right") & df["discount"].notna()).mean()),
}
nulls = {c: int(n) for c, n in df.isna().sum().items() if n > 0}
print(json.dumps({"qa": qa, "nulos_por_columna": nulls}, indent=2, ensure_ascii=False))

{
  "qa": {
    "venta_diaria_distinta_de_suma_horaria": 0.0,
    "stock_6_22_fuente_distinto_de_recuento_propio": 0.0,
    "descuento_fuera_de_(0,1]": 0.003571777777777778
  },
  "nulos_por_columna": {}
}


In [11]:
df = df.sort_values(KEY + ["date"]).reset_index(drop=True)

report.update({
    "rows_out": len(df),
    "date_min": str(df["date"].min().date()),
    "date_max": str(df["date"].max().date()),
    "series": int(df.groupby(KEY).ngroup().nunique()),
    "qa": qa,
    "nulls_per_column": nulls,
})

silver_path = SILVER / "train_silver.parquet"
df.to_parquet(silver_path, index=False)
(SILVER / "train_silver_report.json").write_text(json.dumps(report, indent=2))

print(f"Silver: {report['rows_in']:,} -> {report['rows_out']:,} filas | "
      f"{report['series']:,} series | {report['date_min']} -> {report['date_max']}")

Silver: 4,500,000 -> 4,500,000 filas | 50,000 series | 2024-03-28 -> 2024-06-25


---
## 3. Gold — el dataset para el modelo

**Qué significa cada fila:** un producto en una tienda en un día **d**, con lo que se sabe
al cierre de ese día, para predecir la venta del día **d+1**.

| Grupo | Columnas | Idea |
|---|---|---|
| Estado del día d | `sales`, `stockout_share_6_21`, `peak_hour`, `discount`, `holiday`, `activity`, clima | Cómo ha ido hoy |
| Histórico | `sales_lag_1`, `sales_lag_7`, `sales_roll_mean_7`, `stockout_share_lag_1` | Ayer, hace una semana y la media reciente |
| Conocido de antemano | `discount_next_day`, `holiday_next_day`, `activity_next_day` | Una tienda sabe con antelación sus promociones y el calendario de mañana |
| Calendario | `dow`, `is_weekend`, `month` | Patrón semanal |
| **Objetivo** | `target_next_day_sales` | Lo que se quiere predecir |
| Grado de censura | `target_next_day_stockout_share` | Si mañana hubo rotura de stock, la venta observada **subestima** la demanda real |

**Dos reglas para no hacer trampas:**
1. Ninguna feature usa información posterior al día d (salvo las `*_next_day`, que son planificadas).
2. Los desfases se calculan por **fecha**, no por posición: si falta un día, el valor queda en `NaN`
   en lugar de coger el de otro día.

In [12]:
df = pd.read_parquet(silver_path)
df["stockout_share_6_21"] = (df["stockout_hours_6_21"] / 16).astype("float32")   # fracción de las 16 horas (6..21) sin stock

def value_at(col, days_back):
    # Valor de `col` en la fecha (date - days_back) de la misma tienda-producto.
    # days_back=1 -> ayer; days_back=-1 -> mañana. Si ese día no existe, NaN.
    other = df[KEY + ["date", col]].copy()
    other["date"] = other["date"] + pd.Timedelta(days=days_back)
    return df[KEY + ["date"]].merge(other, on=KEY + ["date"], how="left")[col].to_numpy()

# Histórico
df["sales_lag_1"] = value_at("sales", 1)
df["sales_lag_7"] = value_at("sales", 7)
last7 = pd.DataFrame({k: value_at("sales", k) for k in range(1, 8)})
df["sales_roll_mean_7"] = last7.mean(axis=1).where(last7.notna().sum(axis=1) >= 3)   # media de los 7 días previos, mínimo 3 datos
df["stockout_share_lag_1"] = value_at("stockout_share_6_21", 1)

# Conocido de antemano sobre mañana
df["discount_next_day"] = value_at("discount", -1)
df["holiday_next_day"] = value_at("holiday", -1)
df["activity_next_day"] = value_at("activity", -1)

# Calendario
df["dow"] = df["date"].dt.dayofweek.astype("int8")
df["is_weekend"] = (df["dow"] >= 5).astype("int8")
df["month"] = df["date"].dt.month.astype("int8")

# Objetivo
df["target_next_day_sales"] = value_at("sales", -1)
df["target_next_day_stockout_share"] = value_at("stockout_share_6_21", -1)

In [13]:
gold_columns = [
    "city_id", "store_id", "management_group_id", "product_id",
    "category_1", "category_2", "category_3", "date",
    "sales", "stockout_share_6_21", "peak_hour",
    "discount", "holiday", "activity",
    "precipitation", "temperature", "humidity", "wind_level",
    "sales_lag_1", "sales_lag_7", "sales_roll_mean_7", "stockout_share_lag_1",
    "discount_next_day", "holiday_next_day", "activity_next_day",
    "dow", "is_weekend", "month",
    "target_next_day_sales", "target_next_day_stockout_share",
]
gold = df[gold_columns]
gold = gold[gold["target_next_day_sales"].notna()].reset_index(drop=True)   # sin objetivo no sirve para entrenar

# Comprobaciones básicas
assert not gold.duplicated(KEY + ["date"]).any(), "Hay filas duplicadas en Gold"
assert (gold["target_next_day_sales"] >= 0).all()

gold_path = GOLD / "train_gold.parquet"
gold.to_parquet(gold_path, index=False)

print(f"Gold: {len(gold):,} filas x {gold.shape[1]} columnas | {gold['date'].min().date()} -> {gold['date'].max().date()}")
print(f"Filas cuyo objetivo tuvo alguna rotura de stock: {(gold['target_next_day_stockout_share'] > 0).mean():.1%}")
gold.head()

Gold: 4,450,000 filas x 30 columnas | 2024-03-28 -> 2024-06-24
Filas cuyo objetivo tuvo alguna rotura de stock: 44.2%


,city_id,store_id,management_group_id,product_id,category_1,category_2,category_3,date,sales,stockout_share_6_21,...,sales_roll_mean_7,stockout_share_lag_1,discount_next_day,holiday_next_day,activity_next_day,dow,is_weekend,month,target_next_day_sales,target_next_day_stockout_share
0,0,0,2,4,29,78,82,2024-03-28,0.5,0.8125,...,NaN,NaN,0.882,0.0,1.0,3,0,3,1.3,0.3125
1,0,0,2,4,29,78,82,2024-03-29,1.3,0.3125,...,NaN,0.8125,0.882,1.0,1.0,4,0,3,5.3,0.0000
2,0,0,2,4,29,78,82,2024-03-30,5.3,0.0000,...,NaN,0.3125,0.879,1.0,1.0,5,1,3,4.2,0.0000
3,0,0,2,4,29,78,82,2024-03-31,4.2,0.0000,...,2.366667,0.0000,0.882,0.0,1.0,6,1,3,0.7,0.0000
4,0,0,2,4,29,78,82,2024-04-01,0.7,0.0000,...,2.825000,0.0000,0.882,0.0,1.0,0,0,4,2.3,0.0000


In [14]:
# Resumen del objetivo y de los huecos (los NaN de los primeros días son normales: no tienen histórico)
display(gold["target_next_day_sales"].describe().to_frame().T)
display(gold.isna().mean().loc[lambda s: s > 0].round(3).to_frame("proporción de nulos"))

,count,mean,std,min,25%,50%,75%,max
target_next_day_sales,4450000.0,1.001769,1.409621,0.0,0.4,0.7,1.1,44.900002


,proporción de nulos
sales_lag_1,0.011
sales_lag_7,0.079
sales_roll_mean_7,0.034
stockout_share_lag_1,0.011


---
## Qué queda para la siguiente fase

- **Partición temporal:** entrenar con las fechas antiguas y validar con las últimas dos semanas, nunca al azar.
- **Baselines a batir:** `sales_lag_1`, `sales_lag_7` y `sales_roll_mean_7`, medidos con **WAPE** y MAE.
- **Roturas de stock:** evaluar por separado los días con y sin rotura (`target_next_day_stockout_share`),
  porque con rotura la venta observada es menor que la demanda real.

**Limitaciones que conviene recordar:** solo 90 días por serie (sin estacionalidad anual),
ventas normalizadas con un coeficiente (no son unidades reales), no hay datos de desperdicio y
los datos son de una plataforma china, no de supermercados de barrio españoles.